<h1>Project - Team 22 </h1>
<h2>Part One<h2>

<div dir="rtl" style="text-align: right;">

<h1>שמות חברי הקבוצה + ת"ז</h1>


<div dir="rtl"> 
רוני פחימה: 212009260
זוהר קולפ: 322435918

<div dir="rtl"> 
<h2>קישור לגיט</h2>
<a href="https://github.com/ZoeyKulp/movie-dataset-TJ-TZ">https://github.com/ZoeyKulp/movie-dataset-TJ-TZ</a>

<div dir="rtl"> 
<h3>סרטים שמתחילים באותיות: TJ-TZ</h3>

<div dir="rtl" style="text-align:right;">

<h2>1. יבוא ספריות</h2>

</div>

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import numpy as np
import time
from urllib.parse import quote

<div dir="rtl" style="text-align:right;">

<h2>2. טעינה וסינון title.basics</h2>

</div>

In [2]:
basics = pd.read_csv("title.basics.tsv.gz", sep="\t", compression="gzip")
basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [3]:
print(basics.dtypes)  ## בדיקת types

tconst            object
titleType         object
primaryTitle      object
originalTitle     object
isAdult            int64
startYear         object
endYear           object
runtimeMinutes    object
genres            object
dtype: object


In [4]:
basics["genres"] = basics["genres"].str.split(",")
basics["genres"].head()

0            [Documentary, Short]
1              [Animation, Short]
2    [Animation, Comedy, Romance]
3              [Animation, Short]
4                         [Short]
Name: genres, dtype: object

In [5]:
# המרת עמודות מטקסט למספר
basics['startYear'] = pd.to_numeric(basics['startYear'], errors='coerce')
basics['runtimeMinutes'] = pd.to_numeric(basics['runtimeMinutes'], errors='coerce')

# סינון לפי דרישות המטלה
basics_filtered = basics[
    (basics["titleType"] == "movie") &
    (basics["startYear"] <= 2024) &
    (basics["runtimeMinutes"] >= 60) &
    (basics["runtimeMinutes"] <= 300) &
    (basics["primaryTitle"].str.upper().str.match(r"^T[J-Z]", na=False))
]

columns_to_keep = ['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']
final_movies = basics_filtered[columns_to_keep]

print(f"סרטים TJ-TZ: {len(final_movies):,}")
final_movies.head()

סרטים TJ-TZ: 7,982


,tconst,primaryTitle,startYear,runtimeMinutes,genres
3436,tt0003471,Traffic in Souls,1913.0,88.0,"[Crime, Drama]"
7365,tt0007464,Tom Brown's Schooldays,1916.0,75.0,[Drama]
8563,tt0008686,Tom Jones,1917.0,70.0,[Comedy]
8579,tt0008702,Trouble Makers,1917.0,70.0,[Drama]
9568,tt0009706,To Hell with the Kaiser!,1918.0,70.0,"[Comedy, Drama, War]"


In [6]:
final_movies = final_movies.copy()

final_movies["startYear"] = final_movies["startYear"].astype(int)
final_movies["runtimeMinutes"] = final_movies["runtimeMinutes"].astype(int)
print(final_movies.dtypes)

tconst            object
primaryTitle      object
startYear          int64
runtimeMinutes     int64
genres            object
dtype: object


<div dir="rtl" style="text-align:right;">

<h2>3. הוספת דירוגים מ-title.ratings</h2>

</div>

In [7]:
ratings = pd.read_csv("title.ratings.tsv.gz", sep="\t", compression="gzip")
ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2213
1,tt0000002,5.5,317
2,tt0000003,6.4,2325
3,tt0000004,5.1,199
4,tt0000005,6.2,3047


In [8]:
print(ratings.dtypes)  ## בדיקת types

tconst            object
averageRating    float64
numVotes           int64
dtype: object


In [9]:
# inner join - שומרים רק סרטים שיש להם דירוג
final_movies = final_movies.merge(ratings, on="tconst", how="inner")

print(f"סרטים עם דירוג: {len(final_movies):,}")
final_movies.head()

סרטים עם דירוג: 5,759


,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0003471,Traffic in Souls,1913,88,"[Crime, Drama]",6.0,797
1,tt0007464,Tom Brown's Schooldays,1916,75,[Drama],5.7,39
2,tt0009706,To Hell with the Kaiser!,1918,70,"[Comedy, Drama, War]",7.2,18
3,tt0009721,Treasure Island,1917,60,"[Adventure, Drama]",6.6,32
4,tt0009726,True Blue,1918,60,"[Drama, Western]",5.0,18


<div dir="rtl" style="text-align:right;">

<h2>4. הוספת שחקנים מ-title.principals</h2>

</div>

In [10]:
principals = pd.read_csv("title.principals.tsv.gz", sep="\t", compression="gzip")
principals.head()

,tconst,ordering,nconst,category,job,characters
0,tt0000001,1,nm1588970,self,\N,"[""Self""]"
1,tt0000001,2,nm0005690,director,\N,\N
2,tt0000001,3,nm0005690,producer,producer,\N
3,tt0000001,4,nm0374658,cinematographer,director of photography,\N
4,tt0000002,1,nm0721526,director,\N,\N


<div dir="rtl" style="text-align:right;">

<h2>5. מסננים category=='actor', ממיינים לפי ordering, ולוקחים עד 5 שחקנים מובילים לכל סרט.</h2>

</div>

In [11]:
actors = principals[principals["category"] == "actor"]

actors = actors[actors["tconst"].isin(final_movies["tconst"])]

actors = actors.sort_values(by=["tconst", "ordering"])

lead_actors = actors.groupby("tconst")["nconst"].apply(list).reset_index()

lead_actors["lead_actors_ids"] = lead_actors["nconst"].apply(lambda x: x[:5])

lead_actors = lead_actors[["tconst", "lead_actors_ids"]]

final_movies = final_movies.merge(lead_actors, on="tconst", how="left")

final_movies[["tconst", "primaryTitle", "lead_actors_ids"]].head()

,tconst,primaryTitle,lead_actors_ids
0,tt0003471,Traffic in Souls,"[nm0877941, nm0601596, nm0920607, nm0146957, n..."
1,tt0007464,Tom Brown's Schooldays,"[nm0171058, nm0072674, nm1173678, nm1173753, n..."
2,tt0009706,To Hell with the Kaiser!,"[nm0335517, nm0335517, nm0839144, nm0770844, n..."
3,tt0009721,Treasure Island,"[nm0139333, nm0674141, nm0765125, nm0582209, n..."
4,tt0009726,True Blue,"[nm0267912, nm0165134, nm0779914, nm0212097, n..."


<div dir="rtl" style="text-align:right;">

<h2>6. דירוג הסרטים לפי numVotes ובחירת 5000 הסרטים המובילים</h2>

</div>

In [12]:
final_movies = final_movies.sort_values(
    by="numVotes",
    ascending=False
).head(5000).reset_index(drop=True)

<div dir="rtl" style="text-align:right;">

<h2>7. השלמת נתונים מ-Wikipedia באמצעות Web Scraping</h2>

</div>

In [13]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0 Safari/537.36"}

In [14]:
OMDB_API_KEY = "11eac899"

<div dir="rtl" style="text-align:right;">

<h3>7.1 פונקציה להמרת נתוני תקציב והכנסות לערכים מספריים</h3>

</div>

In [15]:
def parse_money(text):
    """ממיר מחרוזת כסף כמו '$120 million' ל-float במיליוני דולר."""
    if not text:
        return np.nan
    text = str(text)
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace(",", "").replace("$", "").strip()
    multiplier = 1000 if "billion" in text.lower() else 1 if "million" in text.lower() else 1/1_000_000
    numbers = re.findall(r"\d+(?:\.\d+)?", text)
    if numbers:
        nums = [float(n) for n in numbers]
        return round(sum(nums) / len(nums) * multiplier, 2)
    return np.nan

<div dir="rtl" style="text-align:right;">

<h3>7.2 פונקציה לאיסוף תיאור הסרט (plot) באמצעות OMDb API</h3>

</div>

In [16]:
def get_omdb_data(imdb_id):
    """מביא plot מ-OMDb API לפי מזהה IMDb."""
    url = f"http://www.omdbapi.com/?i={imdb_id}&apikey={OMDB_API_KEY}"
    try:
        response = requests.get(url)
        data = response.json()
        if data.get("Response") == "True":
            plot = data.get("Plot")
            return plot if plot and plot != "N/A" else np.nan
    except:
        pass
    return np.nan

<div dir="rtl" style="text-align:right;">

<h3>7.3 מוודא שהדף נמצא הוא אכן דף סרט לפי מילות מפתח ב-infobox</h3>

</div>

In [17]:
def is_movie_page(soup):
    
    infobox = soup.find("table", class_=lambda c: c and "infobox" in c)
    
    if infobox:
        infobox_text = infobox.get_text(" ", strip=True).lower()
        
        movie_words = [
            "directed by",
            "produced by",
            "starring",
            "release date",
            "running time",
            "budget",
            "box office"
        ]
        
        for word in movie_words:
            if word in infobox_text:
                return True
    
    return False

<div dir="rtl" style="text-align:right;">

<h3>7.4  מטפל בדפי disambiguation - מוצא את הקישור הנכון לפי שם הסרט ושנה</h3>

</div>

In [18]:
def find_disambiguation_movie_link(soup, title, year):
    
    if not year:
        return None
    
    target_text = f"{title} ({int(year)} film)".lower()
    
    for link in soup.find_all("a"):
        link_text = link.get_text(" ", strip=True).lower()
        
        if link_text == target_text:
            href = link.get("href")
            
            if href and href.startswith("/wiki/"):
                return "https://en.wikipedia.org" + href
    
    return None

<div dir="rtl" style="text-align:right;">

<h3>7.5  מחפש את דף הסרט בויקיפדיה, מתמודד עם תווים מיוחדים בכותרות, וחולץ את הנתונים מה-infobox</h3>

</div>

In [19]:
def get_wikipedia_data(title, year=None):
    
    info = {
        "Language": np.nan,
        "Country": np.nan,
        "budget": np.nan,
        "BoxOffice": np.nan,
        "plot": np.nan,
        "wiki_url": np.nan,
        "wiki_page_title": np.nan
    }
    
    queries = []
    
    if year:
        queries.append(f"{title} {int(year)} film")
        queries.append(f"{title} film {int(year)}")
    
    queries.append(f"{title} film")
    queries.append(title)
    
    try:
        for query in queries:
            
            search_url = f"https://en.wikipedia.org/w/index.php?search={quote(query)}&ns0=1"
            response = requests.get(search_url, headers=HEADERS)
            
            if response.status_code != 200:
                continue
            
            soup = BeautifulSoup(response.content, "html.parser")
            
            results = soup.find_all("div", class_="mw-search-result-heading")
            
            candidate_urls = []
            
            if results:
                for result in results[:5]:
                    link = result.find("a")
                    if link:
                        candidate_urls.append("https://en.wikipedia.org" + link["href"])
            else:
                candidate_urls.append(response.url)
            
            for page_url in candidate_urls:
                
                response_page = requests.get(page_url, headers=HEADERS)
                
                if response_page.status_code != 200:
                    continue
                
                soup_page = BeautifulSoup(response_page.content, "html.parser")
                
                disambig_url = find_disambiguation_movie_link(soup_page, title, year)
                
                if disambig_url:
                    page_url = disambig_url
                    response_page = requests.get(page_url, headers=HEADERS)
                    
                    if response_page.status_code != 200:
                        continue
                    
                    soup_page = BeautifulSoup(response_page.content, "html.parser")
                
                if not is_movie_page(soup_page):
                    continue
                
                info["wiki_url"] = page_url
                
                page_title = soup_page.find("h1")
                if page_title:
                    info["wiki_page_title"] = page_title.get_text(" ", strip=True)
                
                infobox = soup_page.find("table", class_=lambda c: c and "infobox" in c)
                
                if infobox:
                    for row in infobox.find_all("tr"):
                        th = row.find("th")
                        td = row.find("td")
                        
                        if not th or not td:
                            continue
                        
                        label = th.get_text(" ", strip=True).lower()
                        value = td.get_text(" ", strip=True)
                        value = re.sub(r"\[.*?\]", "", value).strip()
                        
                        if label in ["language", "languages"]:
                            info["Language"] = value
                        
                        elif label in ["country", "countries"]:
                            info["Country"] = value
                        
                        elif "budget" in label:
                            info["budget"] = parse_money(value)
                        
                        elif "box office" in label or "gross" in label:
                            info["BoxOffice"] = parse_money(value)
                
                return info
    
    except Exception as e:
        return info
    
    return info

In [21]:
wiki_results = []

# לוקח את כל הסרטים מתוך final_movies
movies_sample = final_movies.reset_index(drop=True)

print("אוסף נתוני Wikipedia עבור כל הסרטים...")

for i, row in movies_sample.iterrows():
    
    wiki_data = get_wikipedia_data(
        row["primaryTitle"],
        row["startYear"]
    )
    
    wiki_data["tconst"] = row["tconst"]
    wiki_data["primaryTitle"] = row["primaryTitle"]
    wiki_data["startYear"] = row["startYear"]
    
    # שליפת plot מ-OMDb
    wiki_data["plot"] = get_omdb_data(row["tconst"])
    
    wiki_results.append(wiki_data)
    
    time.sleep(0.3)

wiki_df1 = pd.DataFrame(wiki_results)

print("סיום!")
wiki_df1.head()

אוסף נתוני Wikipedia עבור כל הסרטים...
סיום!


,Language,Country,budget,BoxOffice,plot,wiki_url,wiki_page_title,tconst,primaryTitle,startYear
0,English,United States,30.0,401.2,A cowboy doll is profoundly jealous when a new...,https://en.wikipedia.org/wiki/Toy_Story,Toy Story,tt0114709,Toy Story,1995
1,English,United States,200.0,1067.0,The toys are mistakenly delivered to a day-car...,https://en.wikipedia.org/wiki/Toy_Story_3,Toy Story 3,tt0435761,Toy Story 3,2010
2,English,United States,173.5,1496.0,The story involves Maverick confronting his pa...,https://en.wikipedia.org/wiki/Top_Gun:_Maverick,Top Gun: Maverick,tt1745960,Top Gun: Maverick,2022
3,English Scots,United Kingdom,1.5,60.0,"Renton, deeply immersed in the Edinburgh drug ...",https://en.wikipedia.org/wiki/Trainspotting_(f...,Trainspotting (film),tt0117951,Trainspotting,1996
4,English,United States,172.5,709.7,An ancient struggle between two Cybertronian r...,https://en.wikipedia.org/wiki/Transformers_200...,Transformers (film),tt0418279,Transformers,2007


<div dir="rtl" style="text-align:right;">

<h2>8. בניית הטבלה הסופית</h2>

</div>

In [25]:
# מיזוג נתוני ויקיפדיה עם טבלת הסרטים
dataset = final_movies.merge(
    wiki_df1[['tconst', 'Language', 'Country', 'budget', 'BoxOffice', 'plot']],
    on='tconst',
    how='left'
)

# סידור עמודות לפי המטלה
dataset = dataset[['tconst', 'primaryTitle', 'startYear', 'genres', 'lead_actors_ids',
                   'runtimeMinutes', 'averageRating', 'Language', 'Country',
                   'numVotes', 'budget', 'BoxOffice', 'plot']]

print(f"גודל הטבלה הסופית: {dataset.shape}")

dataset.head()

גודל הטבלה הסופית: (5000, 13)


,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,Language,Country,numVotes,budget,BoxOffice,plot
0,tt0114709,Toy Story,1995,"[Adventure, Animation, Comedy]","[nm0000158, nm0000741, nm0725543, nm0001815, n...",81,8.3,English,United States,1172423,30.0,401.2,A cowboy doll is profoundly jealous when a new...
1,tt0435761,Toy Story 3,2010,"[Adventure, Animation, Comedy]","[nm0000158, nm0000741, nm0000885, nm0725543, n...",103,8.3,English,United States,968594,200.0,1067.0,The toys are mistakenly delivered to a day-car...
2,tt1745960,Top Gun: Maverick,2022,"[Action, Drama]","[nm0000129, nm1886602, nm0000174, nm1506981, n...",130,8.2,English,United States,857407,173.5,1496.0,The story involves Maverick confronting his pa...
3,tt0117951,Trainspotting,1996,"[Crime, Drama]","[nm0000191, nm0001971, nm0001538, nm0571727, n...",93,8.1,English Scots,United Kingdom,768535,1.5,60.0,"Renton, deeply immersed in the Edinburgh drug ..."
4,tt0418279,Transformers,2007,"[Action, Adventure, Sci-Fi]","[nm0479471, nm0241049, nm0879085, nm0026364, n...",144,7.1,English,United States,717934,172.5,709.7,An ancient struggle between two Cybertronian r...


In [29]:
print(dataset.dtypes) ## בדיקת types

tconst              object
primaryTitle        object
startYear            int64
genres              object
lead_actors_ids     object
runtimeMinutes       int64
averageRating      float64
Language            object
Country             object
numVotes             int64
budget             float64
BoxOffice          float64
plot                object
dtype: object


<div dir="rtl" style="text-align:right;">

<h2>9. ניתוח ערכים חסרים</h2>

</div>

In [26]:
# ניתוח ערכים חסרים עבור הטבלה הסופית
missing_summary = pd.DataFrame({
    "Missing Count": dataset.isna().sum(),
    "Missing Percent": dataset.isna().mean() * 100
})

missing_summary

,Missing Count,Missing Percent
tconst,0,0.00
primaryTitle,0,0.00
startYear,0,0.00
genres,0,0.00
lead_actors_ids,486,9.72
runtimeMinutes,0,0.00
averageRating,0,0.00
Language,1196,23.92
Country,1183,23.66
numVotes,0,0.00


<div dir="rtl" style="text-align:right;">

<h2>10. שמירה ל-CSV</h2>

</div>

In [28]:
dataset.to_csv("dataset.csv", index=False, encoding='utf-8-sig')
print("הקובץ נשמר: dataset.csv")
print(f"סה\"כ סרטים: {len(dataset):,}")

הקובץ נשמר: dataset.csv
סה"כ סרטים: 5,000
